# Model A — independent YOLO26 six-class baseline

**Knowledge-Preserving Vocabulary Expansion for Frame-Based BISINDO Classification: A Comparative Study of Independent and Sequential YOLO26 Training**

## 1. Research objective and Model A definition
Model A independently fine-tunes the official ImageNet-pretrained `yolo26n-cls.pt` on saya, kamu, senang, bingung, marah, apa_kabar. One selected full-context RGB frame per included source video. No temporal translation, Model B/C, architectural expansion, tuning or Kaggle evaluation.

**Construction status:** implemented pipeline; official training NOT RUN; final metrics NOT AVAILABLE. Existing selected-frame coverage is incomplete (480/1,800) and semantics unassessed. Official preparation fails closed until all source videos have explicit, traceable include/exclude decisions. A successful technical smoke does not resolve this scientific blocker.

Read `reports/AUDIT_AND_USAGE.md` and `../../old hypothesa/ARCHIVE_MANIFEST.md` (archive is at repository root).

## 2. Environment inspection
Use the existing `.venv-bisindo` kernel. The system Python lacks Ultralytics. This workflow checks the audited Ultralytics 8.4.197 API; re-audit before changing that version.

In [ ]:
from pathlib import Path
import sys, json
# Works from repository root or this notebook's folder; no fixed machine path.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'experiments/model_a/model_a.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open from repository root or experiments/model_a')
WORK = ROOT / 'experiments/model_a'
sys.path.insert(0, str(WORK))
import model_a as a
from IPython.display import display
import pandas as pd
ENVIRONMENT = a.environment()
a.write_json(WORK / 'reports/environment.json', ENVIRONMENT)
display(ENVIRONMENT)

## 3. Configuration and random seeds
Execution flags default to **False**. Set exactly one True to run a fit. Smoke artifacts never enter official run folders. Parameter values are initial design choices, not demonstrated optima. Deterministic algorithms use warning mode where a deterministic implementation is unavailable; exact cross-hardware reproducibility is not guaranteed.

In [ ]:
RUN_TRAINING = False  # Explicit opt-in: full official Model A; requires completed frame decisions.
RUN_SMOKE_TEST = False  # Explicit opt-in: 1 epoch, 2 existing frames/class/split; technical only.
LOAD_PRETRAINED = True  # Downloads and verifies official asset; never starts training.
assert not (RUN_TRAINING and RUN_SMOKE_TEST), 'Choose one execution mode'
CONFIG = json.loads((WORK / 'config/model_a.json').read_text())
a.seeds(CONFIG['training']['seed'])
IMG_SIZE = CONFIG['training']['imgsz']  # Initial 224; edit config for a separately recorded experiment.
display(CONFIG)

## 4. Dataset discovery and source manifests
Raw internal videos: `04_dataset/train/`. Frozen identities: `02_exp/arsip_pipeline/experiment_yolo26_v3/split_lock.csv`. No split is regenerated. Internal inventory must contain 2,400 videos (300/class), six-class subset 1,800.

Existing seed42 development **selection manifest only** supplies candidate/frame identities; no weights, scores, hyperparameter search results or historical training configuration are read. Missing selections remain missing. Cached source JPEGs retain full source-frame context. `frame_candidates.py` and `pilot_study.py` contain historical extraction/selection implementations; they are audited, not invoked here.

In [ ]:
AUDIT, FRAME_TEMPLATE = a.audit(hash_videos=True)
SOURCE = a.read_csv(WORK / 'manifests/source_inventory.csv')
display(pd.read_csv(WORK / 'reports/split_counts.csv'))
display({k:v for k,v in AUDIT.items() if k != 'protected_input_hashes'})

## 5. Frozen-split verification and leakage checks
The audit validates unique IDs, paths, video hashes, canonical labels, source/split identity, physical internal inventory and frozen inventory equality. All 1,800 source files are hashed. Identical source bytes across train/val are rejected; included selected JPEG bytes and decoded RGB are checked again during preparation. This does not prove absence of near-duplicates or overlapping video segments.

Internal validation is **not signer-independent**. Same researcher-signers may occur in both splits. Historical Kaggle rows are read only as frozen identity metadata, never as training/validation samples or preprocessing targets.

In [ ]:
assert len(SOURCE) == 1800 and set(SOURCE.class_name) == set(a.CLASSES)
assert set(SOURCE.source) == {'train'} and set(SOURCE.split) == {'train', 'val'}
assert not set(SOURCE[SOURCE.split.eq('train')].video_sha256) & set(SOURCE[SOURCE.split.eq('val')].video_sha256)
assert a.sha(a.LOCK) == AUDIT['protected_input_hashes'][str(a.LOCK.relative_to(ROOT))]
print('Frozen identity/source checks PASS. Signer/near-duplicate independence UNVERIFIED.')

## 6. Six-class dataset preparation
`manifests/frame_decisions.csv` accounts for every source video. The audit creates it once, never overwrites researcher edits. Template: 480 existing selections; 1,320 missing. Do not choose frames using model accuracy.

Before official fitting, complete selections using a documented approved selection/review policy and the existing candidate catalogue. Bind `candidate_id`, `frame_index`, `source_frame_path`, `source_frame_sha256` to that catalogue. Each row needs `disposition=include|exclude`, `reviewer`, `decision_note`. Included rows require `semantic_status=valid_for_target` and `reviewed_candidate_id=candidate_id`. Retain difficult valid frames; exclusions need scientific reasons, not performance. Review historical concerns at video level without assuming a different frame shares the same adjudication.

Preparation rejects unresolved decisions, frozen identity edits, invalid/ambiguous included frames and cross-split selected-content duplicates. Exclusions, coverage and all 1,800 decisions remain traceable. Immutable dataset directories use content fingerprints. Future comparisons must reuse this frozen Model A release, including any exclusions.

In [ ]:
DECISIONS = a.read_csv(a.DECISIONS)
display(DECISIONS.groupby(['class_name', 'disposition']).size().unstack(fill_value=0))
CONCERNS = pd.read_csv(WORK / 'reports/historical_review_concerns.csv')
display(CONCERNS.head(12))
DATASET = None
if RUN_TRAINING:
    DATASET, PREPARED_ROWS, DATASET_FINGERPRINT = a.prepare(smoke=False)
elif RUN_SMOKE_TEST:
    DATASET, PREPARED_ROWS, DATASET_FINGERPRINT = a.prepare(smoke=True)
else:
    print('Preparation preview only. Official training remains blocked until frame decisions are complete.')

## 7. Image preprocessing and visual sanity checks
Installed default training uses RandomResizedCrop; default validation uses Resize + CenterCrop. Both can remove gesture regions. The small `ModelATrainer`, `ModelAValidator` and `ModelAPredictor` overrides share `LetterboxRGB` instead.

**Every enabled transform:** convert PIL image to RGB; bilinear aspect-preserving resize; centered constant `(114,114,114)` padding to 224²; float tensor /255; normalize mean=(0,0,0), std=(1,1,1), matching this Ultralytics pretrained pipeline. No random augmentation: no flips, rotation, crop, color jitter, perspective, erasing, MixUp or CutMix. Validation/inference are deterministic. Resizing cannot restore context already missing from the source frame.

[Official Ultralytics classification documentation](https://docs.ultralytics.com/tasks/classify/). The following images are an inspection aid, **not** semantic approval.

In [ ]:
figure = a.sanity_plot(FRAME_TEMPLATE, IMG_SIZE)
display(figure)
import matplotlib.pyplot as plt
plt.close(figure)

## 8. Official pretrained YOLO26 initialization
Fresh asset from Ultralytics/assets v8.4.0. SHA256 pinned and verified before loading; task must be classify with 1,000 ImageNet outputs. Historical checkpoints are never searched. Network failure is a real blocker; no fallback to trained weights.

In [ ]:
PRETRAINED_PATH = None
PRETRAINED_MODEL = None
if LOAD_PRETRAINED:
    PRETRAINED_PATH, PRETRAINED_MODEL = a.pretrained()
    display(json.loads((WORK / 'reports/pretrained_verification.json').read_text()))
else:
    print('Pretrained loading explicitly skipped; fit still verifies the official asset.')

## 9. Model architecture summary
The official pretrained network initially has 1,000 outputs. Standard ClassificationTrainer adapts its classifier to six outputs during fine-tuning. No other architecture modification. Actual trained architecture exports to `fit/six_class_architecture.txt`; until fitting it is NOT AVAILABLE.

In [ ]:
if PRETRAINED_MODEL is not None:
    PRETRAINED_MODEL.info()
    print('Full pretrained architecture:', WORK / 'reports/pretrained_architecture.txt')
    del PRETRAINED_MODEL
    PRETRAINED_MODEL = None
print('Planned training head: six classes; actual output mapping is inspected after fitting.')

## 10. Training configuration
SGD; initial LR 0.01; momentum 0.937; weight decay 0.0005; nominal batch 64 with Ultralytics accumulation/weight-decay scaling; actual minibatch fixed 16. Linear LR schedule to factor 0.01; 3 warmup epochs, initial warmup momentum 0.8 and bias LR 0.1. Actual optimizer groups/LRs exported, including any framework parameter-group details. FP32 (AMP disabled); workers=0 for Windows/notebook reliability; seed42; GPU0 when available, otherwise CPU.

Maximum100 epochs, patience20. Best checkpoint uses internal-validation **top-1 accuracy** (last tied best), via validator fitness override. No hyperparameter optimization. Each run records unique UTC microsecond timestamp + seed + dataset fingerprint. No resume or overwrite. Smoke differs only in 1 epoch, patience1 and a 24-frame technical subset.

In [ ]:
display(pd.Series(CONFIG['training'], name='initial_configuration'))
print('Enabled augmentation: none. Deterministic full-frame letterbox only.')

## 11. Model A training
Change the execution flag in section3 and restart/run all. An incomplete official dataset raises an actionable error; do not bypass it. Runs export their own config, environment, code hashes, dataset release, mappings and metrics.

In [ ]:
RUN_DIRECTORY = None
if RUN_TRAINING or RUN_SMOKE_TEST:
    RUN_DIRECTORY = a.run(smoke=RUN_SMOKE_TEST)
    print('Completed run:', RUN_DIRECTORY)
else:
    print('TRAINING NOT RUN. Both execution flags are False.')

## 12. Training curves and early stopping
History comes from this run only. Best epoch reports the last maximum validation top-1, matching tied checkpoint replacement. Early stopping is distinguished from the epoch limit; unavailable framework stop details are labelled rather than invented.

In [ ]:
if RUN_DIRECTORY is not None:
    HISTORY = pd.read_csv(RUN_DIRECTORY / 'training_history.csv')
    curve_columns = [c for c in HISTORY if c in ['train/loss', 'val/loss', 'metrics/accuracy_top1']]
    HISTORY.plot(x='epoch', y=curve_columns, subplots=True, figsize=(9, 7), grid=True)
    plt.tight_layout()
    plt.savefig(RUN_DIRECTORY / 'training_curves.png', dpi=120)
    plt.show()
    EXECUTION = json.loads((RUN_DIRECTORY / 'run_manifest.json').read_text())
    display({k:EXECUTION.get(k) for k in ['mode', 'completed_epochs', 'best_epoch', 'stopping_reason', 'early_stopper_best_epoch']})
else:
    print('Training curves, completed epochs and stopping information: NOT AVAILABLE.')

## 13. Internal validation evaluation
Only the newly trained best checkpoint and internal val split are evaluated. Explicit predictor enforces the same RGB letterbox after reload. Top1, macro precision/recall/F1 and per-class precision/recall/F1/support come from canonical-ID predictions. Confusion rows=true, columns=predicted. Undefined precision/recall/F1 denominators yield0; macro-F1 is the arithmetic mean of **six per-class F1 scores**, not F1 of averaged precision/recall. No detection mAP.

Standalone use: `a.evaluate(best_checkpoint, frozen_rows, new_output_directory, size=224, batch=16, device=0)` or `YOLO(checkpoint).predict(paths, predictor=a.ModelAPredictor, imgsz=224)`. Keep `model_a.py` importable for transform deserialization. Plain default `YOLO.predict` is not the supported inference contract.

In [ ]:
if RUN_DIRECTORY is not None:
    display(json.loads((RUN_DIRECTORY / 'validation_metrics.json').read_text()))
    if RUN_SMOKE_TEST:
        print('TECHNICAL SMOKE ONLY. Official Model A metrics: NOT AVAILABLE.')
else:
    print('Official Model A validation metrics: NOT AVAILABLE. No fit was run.')

## 14. Confusion matrix and per-class errors
Actual output folder names are alphabetical, not research order. `label_mappings.json` records both orders and the output-to-canonical map. Sample exports include IDs, names, canonical labels, model output index, confidence, both full probability vectors and split. Error CSV preserves every mistaken validation sample; no post-hoc label changes.

In [ ]:
if RUN_DIRECTORY is not None:
    display(pd.read_csv(RUN_DIRECTORY / 'per_class_metrics.csv'))
    display(pd.read_csv(RUN_DIRECTORY / 'confusion_matrix.csv', index_col=0))
    display(pd.read_csv(RUN_DIRECTORY / 'validation_errors.csv').head(20))
    from IPython.display import Image as DisplayImage
    display(DisplayImage(filename=str(RUN_DIRECTORY / 'confusion_matrix.png')))
    display(json.loads((RUN_DIRECTORY / 'label_mappings.json').read_text()))
else:
    print('Confusion matrix and error analysis: NOT AVAILABLE.')

## 15. Checkpoint and reproducibility manifest export
Each run retains best.pt and last.pt for later authorized expansion work, together with SHA256 hashes. `run_manifest.json`, `dataset_fingerprint.json`, immutable dataset manifest, explicit exclusions, `actual_training_config.json`, epoch trace, history, architecture and environment bind the run. Do not promote smoke weights to Model A. A failed run is preserved with FAILED status.

In [ ]:
if RUN_DIRECTORY is not None:
    display(json.loads((RUN_DIRECTORY / 'run_manifest.json').read_text()))
else:
    a.write_json(WORK / 'reports/official_results_status.json', {
        'status': 'NOT AVAILABLE', 'official_training_executed': False,
        'reason': 'No official run requested; full selected-frame/review coverage incomplete',
        'metrics': None, 'best_checkpoint': None})
    print('Construction evidence exported. Official checkpoints and metrics: NOT AVAILABLE.')

## 16. Actual results and limitations
No official metrics exist until the full official procedure completes. Technical smoke results, if requested, establish execution only. Existing split: 1,440 train +360 validation across six classes; no new random split. Complete frame decisions remain a methodological blocker; missing selections are not automatically generated and difficult examples are not silently discarded.

Internal validation may share researcher-signers and backgrounds with training. Hash checks detect exact content, not all temporal/near-duplicate overlap (historical senang223/224 share candidate frames within train). Semantic review, signer/session provenance and final frame-policy coverage remain separate requirements. Prior exploratory experiments are historical context only. Kaggle is reserved; no external images/scores used here. No temporal translation or measured real-time claim.

In [ ]:
print('Scope: Model A only; internal validation only.')
print('Run:', str(RUN_DIRECTORY) if RUN_DIRECTORY else 'NOT EXECUTED')
print('Official results:', 'AVAILABLE for this run' if RUN_TRAINING and RUN_DIRECTORY else 'NOT AVAILABLE')
print('Archive:', ROOT / 'old hypothesa/ARCHIVE_MANIFEST.md')